# Index M&A Report Generation

This notebook demonstrates how to generate M&A (Mergers & Acquisitions) analysis reports for a portfolio of stocks.

## Workflow Overview

1. **Configure Environment**: Set up API keys and initialize services
2. **Define Tickers**: Specify the stocks to analyze
3. **Search M&A News**: Fetch M&A-related news using Bigdata.com API
4. **Generate Briefs**: Create executive summaries from news articles
5. **Generate M&A Report**: Produce structured deal analysis


## Step 1: Environment Setup

Load environment variables and initialize required services.


In [2]:
import os
import sys
import json
import asyncio
from pathlib import Path
from datetime import datetime, timedelta, timezone
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

# Verify API keys are set
openai_key = os.getenv('OPENAI_API_KEY')
bigdata_key = os.getenv('BIGDATA_API_KEY')

print(f"OpenAI API Key: {'✓ Set' if openai_key else '✗ Missing'}")
print(f"Bigdata API Key: {'✓ Set' if bigdata_key else '✗ Missing'}")

if not openai_key or not bigdata_key:
    print("\n⚠️  Please create a .env file with your API keys:")
    print("   OPENAI_API_KEY=your_key_here")
    print("   BIGDATA_API_KEY=your_key_here")


OpenAI API Key: ✓ Set
Bigdata API Key: ✓ Set


## Step 2: Initialize Services

Set up the topic search service and report service.


In [3]:
from services.topic_search_service import TopicSearchService
from services.report_service import ReportService
from config.topics import MA_TOPICS

# Initialize services
topic_search_service = TopicSearchService(api_key=bigdata_key)
report_service = ReportService()

print(f"Topic Search Service: ✓ Initialized")
print(f"Report Service: ✓ Initialized (provider: {report_service.llm_service.provider_name})")
print(f"\nM&A Topics loaded: {len(MA_TOPICS)} topics")
for i, topic in enumerate(MA_TOPICS, 1):
    print(f"  {i}. {topic['topic_name']}: {topic['topic_text'][:60]}...")


Topic Search Service: ✓ Initialized
Report Service: ✓ Initialized (provider: openai)

M&A Topics loaded: 4 topics
  1. M&A Activity: What material acquisition, merger, or divestiture activities...
  2. M&A Activity: Is {company} being acquired or in merger discussions with an...
  3. M&A Activity: What acquisition offers or takeover bids has {company} recei...
  4. M&A Activity: What strategic alternatives or sale processes is {company} e...


## Step 3: Configure Analysis Parameters

Define the tickers to analyze and the lookback period.


In [4]:
import pandas as pd
# Configuration
#TICKERS = ["MSFT", "NFLX", "GOOGL", "AAPL"]  # Modify as needed

# Read entity IDs from US_500.csv (column is RP_ENTITY_ID, not Symbol)
df = pd.read_csv('US_5.csv')
print(f"CSV columns: {df.columns.tolist()}")

# Use RP_ENTITY_ID column
TICKERS = df['RP_ENTITY_ID'].tolist()

# print ticker length
print("Total entity IDs:", len(TICKERS))

LOOKBACK_DAYS = 90  # Number of days to search for news

# Calculate date range
end_date = datetime.now(timezone.utc)
start_date = end_date - timedelta(days=LOOKBACK_DAYS)

print(f"Analysis Configuration:")
print(f"  Tickers: {', '.join(TICKERS)}")
print(f"  Period: {start_date.strftime('%Y-%m-%d')} to {end_date.strftime('%Y-%m-%d')}")
print(f"  Lookback: {LOOKBACK_DAYS} days")


CSV columns: ['RP_ENTITY_ID']
Total entity IDs: 6
Analysis Configuration:
  Tickers: DB06B0, C951A2, 8AM205, 94208D, ADF092, 873DB9
  Period: 2025-09-17 to 2025-12-16
  Lookback: 90 days


In [5]:
# Set logger INFO level and to print to console
import logging
logger = logging.getLogger(__name__)

logger.addHandler(logging.StreamHandler())


## Step 4: Search M&A News for Each Ticker

Fetch M&A-related news articles using the topic search service.


In [6]:
import asyncio
from typing import List, Dict
import time

async def search_ma_news_for_ticker(ticker: str, days: int, topic_search_service, custom_topics):
    """Search for M&A news for a single ticker."""
    print(f"\nSearching M&A news for {ticker}...")
    try:
        results = await topic_search_service.search_ticker(
            ticker=ticker,
            days=days,
            custom_topics=custom_topics
        )
        print(f"  ✓ {ticker}: Search completed")
        return ticker, results
                   
    except Exception as e:
        print(f"  ✗ {ticker}: Error - {e}")
        return ticker, None


async def search_ma_news_parallel(tickers: List[str], days: int, max_concurrent: int = 10):
    """Search for M&A news for all tickers in parallel."""
    
    # Create semaphore to limit concurrent tasks
    semaphore = asyncio.Semaphore(max_concurrent)
    
    async def process_with_semaphore(ticker: str):
        """Process a ticker with semaphore to limit concurrency."""
        async with semaphore:
            return await search_ma_news_for_ticker(
                ticker, 
                days, 
                topic_search_service, 
                MA_TOPICS
            )
    
    print(f"Searching M&A news for {len(tickers)} tickers with {max_concurrent} parallel workers...")
    
    # Create tasks for all tickers
    tasks = [process_with_semaphore(ticker) for ticker in tickers]
    
    # Run all tasks concurrently
    results = await asyncio.gather(*tasks, return_exceptions=True)
    
    # Build results dictionary
    all_results = {}
    for result in results:
        if isinstance(result, Exception):
            print(f"  ✗ Task failed with exception: {result}")
        elif isinstance(result, tuple) and len(result) == 2:
            ticker, ticker_results = result
            all_results[ticker] = ticker_results
    
    return all_results


# Run the search in parallel
start_time = time.time()

search_results = await search_ma_news_parallel(TICKERS, LOOKBACK_DAYS, max_concurrent=10)

elapsed_time = time.time() - start_time
print(f"\n{'='*50}")
print(f"Search complete for {len(TICKERS)} tickers")
print(f"Time taken: {elapsed_time/60:.2f} minutes ({elapsed_time:.1f} seconds)")
print(f"Average time per ticker: {elapsed_time/len(TICKERS):.1f} seconds" if TICKERS else "")
print(f"\nResults summary:")
successful = sum(1 for v in search_results.values() if v is not None)
failed = len(search_results) - successful
print(f"  ✓ Successful: {successful}")
print(f"  ✗ Failed: {failed}")

Searching M&A news for 6 tickers with 10 parallel workers...

Searching M&A news for DB06B0...

Searching M&A news for C951A2...

Searching M&A news for 8AM205...

Searching M&A news for 94208D...

Searching M&A news for ADF092...

Searching M&A news for 873DB9...
  ✓ 8AM205: Search completed
  ✓ C951A2: Search completed
  ✓ 873DB9: Search completed
  ✓ ADF092: Search completed
  ✓ 94208D: Search completed
  ✓ DB06B0: Search completed

Search complete for 6 tickers
Time taken: 0.22 minutes (13.1 seconds)
Average time per ticker: 2.2 seconds

Results summary:
  ✓ Successful: 6
  ✗ Failed: 0


In [7]:
# print length of search_results
print(len(search_results))
# print each item topic_results for first item in search_results in json format
print(json.dumps(search_results[TICKERS[0]].get('topic_results', []), indent=2))

    # Save to file
output_dir = Path('output')
output_dir.mkdir(exist_ok=True)

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    
# Save search_results to json file so that it can be re-used
search_results_file = output_dir / f'search_results_{timestamp}.json'
with open(search_results_file, 'w') as f:
    json.dump(search_results, f, indent=2)

# print file saved at path
print(f"File saved at path: {f.name}")



6
[
  {
    "id": "5DDA872D32E026FBE74CC7B0646E6CEC",
    "headline": "Electronic Arts forecasts softer outlook amid record buyout scrutiny",
    "timestamp": "2025-12-16T01:11:26",
    "time_ago": "16h ago",
    "source": "Alliance News",
    "summary": "EA disclosed in September it had agreed to be acquired by Oak-Eagle AcquireCo Inc, leaving the company as a wholly owned unit of a consortium including Silver Lake and Affinity.\nIt filed a definitive ...",
    "full_text": "EA disclosed in September it had agreed to be acquired by Oak-Eagle AcquireCo Inc, leaving the company as a wholly owned unit of a consortium including Silver Lake and Affinity.\nIt filed a definitive proxy with the Securities and Exchange Commission in November, after which shareholder lawsuits in New York and California alleged omissions in the materials. EA also received demand letters from other investors.",
    "document_url": null,
    "relevance": 0.9543776224918538,
    "document_type": "news",
    "search

## Step 5: Generate Executive Briefs

Create concise executive briefs from the news articles.


In [8]:
import asyncio
from typing import List, Dict

async def generate_briefs_for_ticker(ticker: str, results: dict, report_service) -> List[Dict]:
    """Generate executive briefs for a single ticker."""
    if not results or not results.get('topic_results'):
        print(f"\n{ticker}: No results to process")
        return []
        
    print(f"\nGenerating briefs for {ticker}...")
    
    try:
        news_response = {
            "ticker": ticker,
            "company_name": results.get('company_name', ticker),
            "topic_results": results.get('topic_results', [])
        }
        
        briefs = await report_service.generate_topic_briefs(news_response)
        
        ticker_briefs = []
        for brief in briefs:
            ticker_briefs.append({
                "company_name": brief.company_name,
                "topic_name": brief.topic_name,
                "bullet_point": brief.bullet_point
            })
        
        print(f"  ✓ {ticker}: Generated {len(briefs)} briefs")
        return ticker_briefs
        
    except Exception as e:
        print(f"  ✗ {ticker}: Error - {e}")
        return []


async def generate_briefs_for_all_parallel(search_results, max_concurrent: int = 10):
    """Generate executive briefs for all tickers in parallel with concurrency limit."""
    
    # Create semaphore to limit concurrent tasks
    semaphore = asyncio.Semaphore(max_concurrent)
    
    async def process_with_semaphore(ticker: str, results: dict):
        """Process a ticker with semaphore to limit concurrency."""
        async with semaphore:
            return await generate_briefs_for_ticker(ticker, results, report_service)
    
    # Create tasks for all tickers
    tasks = [
        process_with_semaphore(ticker, results)
        for ticker, results in search_results.items()
    ]
    
    print(f"Processing {len(tasks)} tickers with {max_concurrent} parallel workers...")
    
    # Run all tasks concurrently
    results = await asyncio.gather(*tasks, return_exceptions=True)
    
    # Flatten results and handle exceptions
    all_briefs = []
    for result in results:
        if isinstance(result, Exception):
            print(f"  ✗ Task failed with exception: {result}")
        elif isinstance(result, list):
            all_briefs.extend(result)
    
    return all_briefs


# Generate briefs in parallel
import time
start_time = time.time()

all_briefs = await generate_briefs_for_all_parallel(search_results, max_concurrent=10)

elapsed_time = time.time() - start_time
print(f"\n{'='*50}")
print(f"Total briefs generated: {len(all_briefs)}")
print(f"Time taken: {elapsed_time/60:.2f} minutes ({elapsed_time:.1f} seconds)")
print(f"Average time per ticker: {elapsed_time/len(search_results):.1f} seconds")

Processing 6 tickers with 10 parallel workers...

Generating briefs for DB06B0...

Generating briefs for C951A2...

Generating briefs for 8AM205...

Generating briefs for 94208D...

Generating briefs for ADF092...

Generating briefs for 873DB9...
  ✓ 94208D: Generated 1 briefs
  ✓ DB06B0: Generated 1 briefs
  ✓ ADF092: Generated 1 briefs
  ✓ 873DB9: Generated 1 briefs
  ✓ C951A2: Generated 1 briefs
  ✓ 8AM205: Generated 1 briefs

Total briefs generated: 6
Time taken: 0.17 minutes (10.0 seconds)
Average time per ticker: 1.7 seconds


In [9]:
# print first 2 items of all_briefs in json format
print(json.dumps(all_briefs[:2], indent=2))

[
  {
    "company_name": "Electronic Arts Inc.",
    "topic_name": "M&A Activity",
    "bullet_point": "* **Electronic Arts Inc.** enters definitive agreement to be acquired by a consortium led by Saudi Arabia's Public Investment Fund, valuing the company at $55 billion, with shareholders receiving $210 per share in cash, a 25% premium on pre-announcement prices, expected to conclude by Q1 of fiscal 2027.  \nSource: https://www.sec.gov/Archives/edgar/data/712515/000162828025047811/ea-20250930.htm"
  },
  {
    "company_name": "Hologic Inc.",
    "topic_name": "M&A Activity",
    "bullet_point": "* **Hologic Inc.** agrees to be acquired by Blackstone and TPG for up to $79 per share, valuing the deal at $18.3 billion; includes a 45-day 'go-shop' period for alternative proposals, expected to close in H1 2026 pending approvals and shareholder consent.  *Source: https://www.businesswire.com/news/home/20251021765122/en/Hologic-to-be-Acquired-by-Blackstone-and-TPG-for-up-to-%2479-per-Share*"

In [10]:
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    
# Save brief result to json file so that it can be re-used
brief_results_file = output_dir / f'search_results_{timestamp}.json'

# Save all_briefs to json file
with open(brief_results_file, 'w') as f:
    json.dump(all_briefs, f, indent=2)

print(f"Brief File saved at path: {f.name}")



Brief File saved at path: output/search_results_20251216_123544.json


## Step 6: Generate Wall Street Desk Notes

Consolidate briefs into Wall Street-style desk notes for each company. This reduces token usage for the final M&A report.



In [11]:
import asyncio
from typing import List, Dict
from datetime import datetime

async def generate_desk_note_for_company(
    company_name: str, 
    company_briefs: List[Dict], 
    system_prompt: str, 
    user_template: str, 
    llm_service
) -> Dict:
    """Generate desk note for a single company."""
    print(f"  Generating desk note for {company_name}...")
    
    # Format briefs for this company
    briefs_text = ""
    for brief in company_briefs:
        briefs_text += f"- company_name: {brief.get('company_name')}\n"
        briefs_text += f"  topic_name: {brief.get('topic_name')}\n"
        briefs_text += f"  bullet_point: {brief.get('bullet_point')}\n\n"
    
    # Render template
    current_datetime = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    user_prompt = user_template.replace('{{current_datetime}}', current_datetime)
    user_prompt = user_prompt.replace('{{briefs}}', briefs_text)
    
    full_prompt = f"{system_prompt}\n\n{user_prompt}"
    
    try:
        desk_note = await llm_service.generate_content_raw(prompt=full_prompt, model=None)
        print(f"    ✓ {company_name}: Generated ({len(company_briefs)} briefs consolidated)")
        return {
            "company_name": company_name,
            "desk_note": desk_note,
            "briefs_count": len(company_briefs)
        }
    except Exception as e:
        print(f"    ✗ {company_name}: Error - {e}")
        return None


async def generate_desk_notes_parallel(briefs, max_concurrent: int = 10):
    """Generate Wall Street desk notes from briefs, grouped by company, in parallel."""
    import yaml
    
    # Group briefs by company
    briefs_by_company = {}
    for brief in briefs:
        company = brief.get('company_name', 'Unknown')
        if company not in briefs_by_company:
            briefs_by_company[company] = []
        briefs_by_company[company].append(brief)
    
    print(f"Generating desk notes for {len(briefs_by_company)} companies with {max_concurrent} parallel workers...")
    
    # Load desk note prompt
    with open('config/prompts.yaml', 'r', encoding='utf-8') as f:
        prompts = yaml.safe_load(f)
    
    prompt_config = prompts.get('wallstreet_desk_note', {})
    system_prompt = prompt_config.get('system_prompt', '')
    user_template = prompt_config.get('user_template', '')
    
    llm_service = report_service.llm_service
    
    # Create semaphore to limit concurrent tasks
    semaphore = asyncio.Semaphore(max_concurrent)
    
    async def process_with_semaphore(company_name: str, company_briefs: List[Dict]):
        """Process a company with semaphore to limit concurrency."""
        async with semaphore:
            return await generate_desk_note_for_company(
                company_name, 
                company_briefs, 
                system_prompt, 
                user_template, 
                llm_service
            )
    
    # Create tasks for all companies
    tasks = [
        process_with_semaphore(company_name, company_briefs)
        for company_name, company_briefs in briefs_by_company.items()
    ]
    
    # Run all tasks concurrently
    results = await asyncio.gather(*tasks, return_exceptions=True)
    
    # Filter out None results and exceptions
    all_desk_notes = []
    for result in results:
        if isinstance(result, Exception):
            print(f"  ✗ Task failed with exception: {result}")
        elif result is not None:
            all_desk_notes.append(result)
    
    return all_desk_notes


# Generate desk notes from briefs in parallel
import time
start_time = time.time()

desk_notes = await generate_desk_notes_parallel(all_briefs, max_concurrent=10)

elapsed_time = time.time() - start_time
print(f"\n{'='*50}")
print(f"Total desk notes generated: {len(desk_notes)}")
print(f"Time taken: {elapsed_time/60:.2f} minutes ({elapsed_time:.1f} seconds)")
print(f"Average time per company: {elapsed_time/len(desk_notes):.1f} seconds" if desk_notes else "")
print(f"\nBreakdown by company:")
for dn in desk_notes:
    print(f"  • {dn['company_name']}: {dn['briefs_count']} briefs consolidated")

Generating desk notes for 6 companies with 10 parallel workers...
  Generating desk note for Electronic Arts Inc....
  Generating desk note for Hologic Inc....
  Generating desk note for Kenvue Inc....
  Generating desk note for Dayforce Inc....
  Generating desk note for Warner Bros Discovery Inc....
  Generating desk note for Becton Dickinson & Co....
    ✓ Hologic Inc.: Generated (1 briefs consolidated)
    ✓ Dayforce Inc.: Generated (1 briefs consolidated)
    ✓ Becton Dickinson & Co.: Generated (1 briefs consolidated)
    ✓ Kenvue Inc.: Generated (1 briefs consolidated)
    ✓ Electronic Arts Inc.: Generated (1 briefs consolidated)
    ✓ Warner Bros Discovery Inc.: Generated (1 briefs consolidated)

Total desk notes generated: 6
Time taken: 0.07 minutes (4.4 seconds)
Average time per company: 0.7 seconds

Breakdown by company:
  • Electronic Arts Inc.: 1 briefs consolidated
  • Hologic Inc.: 1 briefs consolidated
  • Kenvue Inc.: 1 briefs consolidated
  • Dayforce Inc.: 1 briefs co

In [12]:
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

desk_notes_file = output_dir / f'desk_notes_{timestamp}.json'
# Save desk_notes to json file
with open(desk_notes_file, 'w') as f:
    json.dump(desk_notes, f, indent=2)

print(f"Desk note saved at path: {f.name}")



Desk note saved at path: output/desk_notes_20251216_123548.json


## Step 7: Generate M&A Report

Generate the final M&A analysis report using the consolidated desk notes (more token-efficient than raw briefs).

In [13]:
import yaml

async def generate_ma_report_from_desk_notes(desk_notes):
    """Generate M&A report from desk notes using the portfolio_report_ma prompt.
    
    Using desk notes instead of raw briefs is more token-efficient for large portfolios.
    """
    
    
    # Load prompt template
    with open('config/prompts.yaml', 'r', encoding='utf-8') as f:
        prompts = yaml.safe_load(f)
    
    prompt_config = prompts.get('portfolio_report_ma', {})
    system_prompt = prompt_config.get('system_prompt', '')
    user_template = prompt_config.get('user_template', '')
    
    # Format desk notes as briefs for the prompt
    # Each desk note represents consolidated M&A info for a company
    briefs_text = ""
    for i, dn in enumerate(desk_notes, 1):
        briefs_text += f"\n{i}. Company: {dn.get('company_name', 'Unknown')}\n"
        briefs_text += f"   Topic: M&A Activity (consolidated from {dn.get('briefs_count', 0)} briefs)\n"
        briefs_text += f"   Brief: {dn.get('desk_note', '')}\n"
    
    # Render template
    current_datetime = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    user_prompt = user_template.replace('{{current_datetime}}', current_datetime)
    user_prompt = user_prompt.replace('{{briefs}}', briefs_text)
    
    full_prompt = f"{system_prompt}\n\n{user_prompt}"
    
    # Generate report
    print(f"Generating M&A report from {len(desk_notes)} desk notes...")
    llm_service = report_service.llm_service
    report_text = await llm_service.generate_content_raw(prompt=full_prompt, model='gpt-5-mini')
    
    return report_text

# Generate the report using desk notes (more token-efficient)
if desk_notes:
    ma_report = await generate_ma_report_from_desk_notes(desk_notes)
    print("✓ M&A Report generated successfully!")
else:
    print("⚠️ No desk notes available to generate report")
    ma_report = None


Generating M&A report from 6 desk notes...
✓ M&A Report generated successfully!


## Step 8: Display and Save Report


In [14]:
from IPython.display import Markdown, display

if ma_report:
    # Display report
    report_header = f"""
# M&A Report

**Period:** {start_date.strftime('%Y-%m-%d')} to {end_date.strftime('%Y-%m-%d')}  
**Generated:** {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}  


---
"""
    display(Markdown(report_header + ma_report))
    

    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    
    output_file = output_dir / f'ma_report_{timestamp}.md'
    with open(output_file, 'w', encoding='utf-8') as f:
        f.write(report_header + ma_report)
    print(f"\n✓ Report saved to: {output_file}")
    
else:
    print("No report to display")



# M&A Report

**Period:** 2025-09-17 to 2025-12-16  
**Generated:** 2025-12-16 12:36:12  


---
| Target Company | Acquirer | Deal Value | Status |
|---------------|----------|------------|--------|
| Electronic Arts Inc. (EA) | Consortium led by Saudi Arabia's Public Investment Fund | $55.0B USD | Pending Regulatory Approval |
| Hologic Inc. (HOLX) | Blackstone and TPG | $18.3B USD | Pending Regulatory Approval |
| Kenvue Inc. (KVUE) | Kimberly‑Clark | $48.7B USD (cash + stock) | Pending Regulatory Approval |
| Dayforce Inc. (DAYF) | Thoma Bravo | Undisclosed ($70 per share cash consideration) | Pending Regulatory Approval |
| Warner Bros Discovery Inc. (WBD) | Netflix | $82.7B USD (cash + stock) | Pending Regulatory Approval |

The period features multiple large announced acquisitions of monitored companies, led by Netflix’s $82.7B cash-and-stock agreement for Warner Bros Discovery, followed by Electronic Arts’ $55.0B PIF‑led takeover and Kimberly‑Clark’s $48.7B merger for Kenvue. Hologic agreed to an $18.3B buyout by Blackstone and TPG, and Dayforce received shareholder approval for a $70-per-share cash acquisition by Thoma Bravo. All listed transactions are under definitive agreement or shareholder approval and remain pending regulatory and customary closing conditions.


✓ Report saved to: output/ma_report_20251216_123612.md
